# 训练下一版（v1 / v2 / v3，或 H3 的对照模型）

用预锁定里写死的 v0 配方训练（Ultralytics 8.4.182，YOLO11s，只改数据、名字、输出目录），导出 ONNX，算哈希。

**先在自己电脑上**：每个要用到的站都已经 `lock`，并且 `python dl.py pack-train --site <站>`，
把生成的 `<站>_train_all.zip` 上传到 Drive 的 `mixed_dataset/sites/`。仓库要已经 push（Colab 会 clone 它，需要里面的 `lock.json`）。

**运行时**：GPU。v0 用 150 轮大约一个多小时。

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/mixed_dataset"
REPO_URL = "https://github.com/xiaolin200206/Snapdurian_deploy.git"
ULTRALYTICS = "8.4.182"

NAME = "v1"            # v1 / v2 / v3；H3 用 S1_s0、S1_s1、R0_s1 这种名字
KIND = "version"       # "version" = 前一版的数据 + 新站；"swap" = H3 等量替换
SITES = ["pahang"]     # KIND="version"：训练集里要加的所有站（v2 就是 ["pahang", "penang"]）
NEW_SITE = "pahang"    # KIND="swap"：新加进来的那一站
BASE_SITES = []        # KIND="swap"：前一版已经包含的站（S1 是 []，S2 是 ["pahang"]）
SEED = 0               # 部署的版本一律 0；H3 的另外两个种子用 1、2

In [ ]:
import os, sys, json, subprocess, shutil, zipfile, hashlib
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("不在 Colab 里")
DRIVE = Path(DRIVE_DIR)
REPO = Path("/content") / REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS}", "onnxruntime", "pyyaml", "onnx", "onnxslim"], check=True)
sys.path.insert(0, str(REPO)); os.chdir(REPO)
PRE = json.loads((REPO / "sequential" / "seq_prelock.json").read_text(encoding="utf-8"))
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""): h.update(b)
    return h.hexdigest()
def unzip_once(zp, dest, marker):
    if (dest / marker).exists():
        print("已解压：", dest / marker); return
    print("解压", zp.name, "→", dest)
    local = Path("/content") / zp.name
    if not local.exists(): shutil.copy2(zp, local)
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local) as z: z.extractall(dest)
unzip_once(DRIVE / "merged_v5.zip", REPO / "data", "merged_v5")
MERGED = REPO / "data" / "merged_v5"
print("merged_v5 val 图片：", len(list((MERGED / "images" / "val").glob("*.*"))))

In [ ]:
need = SITES if KIND == "version" else BASE_SITES + [NEW_SITE]
for s in need:
    unzip_once(DRIVE / "sites" / f"{s}_train_all.zip", REPO / "data" / "sites" / s / "train_all", "images")
if KIND == "version":
    cmd = [sys.executable, "dl.py", "build-train", "--name", NAME, "--merged", str(MERGED), "--seed", str(SEED), "--sites", *SITES]
else:
    shutil.copy2(DRIVE / "durian_11cls_640_v5_manifest.csv", "/content/manifest.csv")
    cmd = [sys.executable, "dl.py", "build-swap", "--name", NAME, "--merged", str(MERGED), "--manifest", "/content/manifest.csv",
           "--new-site", NEW_SITE, "--seed", str(SEED)] + (["--base-sites", *BASE_SITES] if BASE_SITES else [])
subprocess.run(cmd, check=True)
kw = json.loads((REPO / "data" / "train" / NAME / "train_kwargs.json").read_text())
print({k: kw[k] for k in ["data", "seed", "name", "epochs", "batch", "imgsz", "optimizer", "lr0"]})

In [ ]:
import torch
assert torch.cuda.is_available(), "没有 GPU：Runtime → Change runtime type → GPU"
from ultralytics import YOLO
model = YOLO(PRE["recipe_for_v1_v3"]["fixed_args"]["model"])
project = DRIVE / "runs_seq"
model.train(**kw, project=str(project), device=0, workers=2)
best = project / NAME / "weights" / "best.pt"
print("best.pt:", best, sha256(best))

In [ ]:
import datetime as dt
onnx_out = Path(YOLO(str(best)).export(format="onnx", opset=12, simplify=True, imgsz=640, batch=1, dynamic=False))
h = sha256(onnx_out)
server_name = f"durian_11cls_640_{NAME}_{dt.date.today():%Y%m%d}_{h[:8]}.onnx"
(DRIVE / "versions").mkdir(exist_ok=True)
shutil.copy2(onnx_out, DRIVE / "versions" / server_name)
print("ONNX SHA-256:", h)
print("已存到 Drive：", DRIVE / "versions" / server_name)
print(f"""
下一步（在自己电脑上）：
  1. 从 Drive 下载 versions/{server_name}
  2. python dl.py register --version {NAME} --onnx <下载的文件> --server-file {server_name}   （只对 v1/v2/v3）
  3. commit + push sequential/versions_log.md —— 在去下一站之前
  4. 上传到服务器 models/，admin.py use {server_name}
H3 的对照模型（S…/R…）不用 register，下一站打分时用 --extra NAME=文件 一起算。""")